#**Prototipo para editar documentos**

___

**Paso 1:** Instalacion de dependencias

In [1]:
!pip install sentence-transformers
!pip install python-docx beautifulsoup4 requests Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

Instalacion de librerias y requerimientos

In [2]:
import re
import os
import io
import requests
from PIL import Image
from sentence_transformers import SentenceTransformer, util
from bs4 import BeautifulSoup
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_BREAK
import torch

- re: Para expresiones regulares, probablemente usado para limpiar texto o buscar patrones.
- os: Para interactuar con el sistema operativo, usado aquí para crear el directorio de salida.
- io: Para trabajar con varios tipos de entrada/salida, usado para manejar datos de imágenes en memoria.
- requests: Para hacer solicitudes HTTP, usado para descargar imágenes desde URLs.
- PIL (Pillow): Para procesamiento de imágenes, usado para abrir, redimensionar y guardar imágenes.
- sentence_transformers: Para generar embeddings de frases y calcular similitud semántica.
- bs4 (BeautifulSoup4): Para parsear contenido HTML.
- docx: Para crear y modificar archivos de Microsoft Word (.docx).
- torch: Un framework de aprendizaje profundo, usado aquí para operaciones con tensores de embeddings de frases.

**Paso 2:** Codigo a ejecutar

In [8]:
# --- Configuration ---
OUTPUT_DIRECTORY = "/content/output_docs"
KEY_PHRASES = [
    "FOTO DE LA PARTE EXTERNA DEL ATM",
    "FOTO DEL ESTADO DE LOS MODULOS",
    "FOTO DEL MODO VENDOR",
    "FOTO DEL ATM EMBALADO",
    "FOTO DEL REPORTE DE SERVICIO"
]
SEMANTIC_SIMILARITY_THRESHOLD = 0.80

# --- CONFIGURACIÓN PARA NEGRITA ---
# Define una lista de palabras o frases que quieres poner en negrita.
# Se han añadido las palabras y frases que solicitaste.
WORDS_TO_BOLD = [
    "INFORME TÉCNICO",
    "Ingeniero de Servicio:",
    "Fecha del Servicio:",
    "Ciudad:",
    "Sitio",
    "Serial",
    "Luno",
    "Cliente",
    "Ticket",
    "ACCIONES REALIZADAS",
    "SOLICITUD DEL CLIENTE",
    "FOTO DE LA PARTE EXTERNA DEL ATM",
    "FOTO DEL ESTADO DE LOS MODULOS",
    "FOTO DEL MODO VENDOR",
    "FOTO DEL ATM EMBALADO",
    "FOTO DEL REPORTE DE SERVICIO"
]
# Pre-compilar los patrones regex para una búsqueda eficiente (si es necesario para flexibilidad)
# Notar que re.escape() es importante si las palabras tienen caracteres especiales de regex.
# Sin embargo, para la función add_paragraph_with_bold_keywords actual, no estamos usando regex aquí.
# Esta línea se mantiene como un recordatorio si se cambia la implementación de la función de negrita.
# Cansao :,c
BOLD_PATTERNS = [re.compile(r'\b' + re.escape(word) + r'\b', re.IGNORECASE) for word in WORDS_TO_BOLD]
# --------------------------------------

# Standard DPI for image processing (common for screen rendering, good for docx)
DPI = 96

# Desired fixed dimensions for full-page images in centimeters
TARGET_IMAGE_WIDTH_CM = 16.26
TARGET_IMAGE_HEIGHT_CM = 23.05

# Convert CM to Inches for docx library
TARGET_IMAGE_WIDTH_INCHES = TARGET_IMAGE_WIDTH_CM / 2.54
TARGET_IMAGE_HEIGHT_INCHES = TARGET_IMAGE_HEIGHT_CM / 2.54

# Define max width for inline images (e.g., half of a typical content width)
INLINE_IMAGE_MAX_WIDTH_INCHES = 3.25

# Heuristic: Minimum dimensions for an image to NOT be considered a placeholder
# (e.g., skips 1x1, 2x2 transparent GIFs often used for spacing)
MIN_IMAGE_DIMENSION_PX = 10 # Images smaller than 10x10 pixels might be skipped


# --- Global Model Initialization ---
sentence_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
key_phrase_embeddings = sentence_model.encode(KEY_PHRASES, convert_to_tensor=True)

# Ensure the output directory exists
os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)


# --- Helper Functions for Text Processing ---
def clean_text_final(text):
    """
    Cleans the end of a string by removing trailing whitespace characters (spaces, tabs, newlines, carriage returns)
    and reduces multiple internal spaces to a single space.
    """
    text = re.sub(r'[\u00A0 \t\n\r ]{1,3}$', '', text)  # Clean trailing whitespace
    text = re.sub(r' {2,}', ' ', text)  # Reduce multiple internal spaces
    return text


def is_key_phrase(text, umbral=SEMANTIC_SIMILARITY_THRESHOLD):
    """
    Checks if the given text semantically matches one of the predefined key phrases.
    """
    cleaned_text = text.strip().upper().replace(" ", " ").replace("\n", " ")
    text_embedding = sentence_model.encode(cleaned_text, convert_to_tensor=True)
    similarities = util.cos_sim(text_embedding, key_phrase_embeddings)
    max_similarity = torch.max(similarities).item()

    if max_similarity >= umbral:
        return True
    return False

# --- NUEVA FUNCIÓN PARA AÑADIR PÁRRAFO CON NEGRITA (SIN CAMBIOS) ---
def add_paragraph_with_bold_keywords(doc, text_content, keywords_to_bold):
    """
    Adds a paragraph to the document, applying bold to specified keywords.
    """
    p = doc.add_paragraph()
    current_pos = 0
    text_lower = text_content.lower() # Para búsquedas insensibles a mayúsculas/minúsculas

    # Ordenar las palabras clave por longitud descendente para manejar superposiciones (opcional pero bueno)
    # Por ejemplo, si tienes "FOTO DEL ATM" y "ATM", quieres que primero detecte la frase más larga.
    sorted_keywords = sorted(keywords_to_bold, key=len, reverse=True)
    lower_sorted_keywords = [k.lower() for k in sorted_keywords]

    # Usar un enfoque basado en regex para encontrar todas las coincidencias y sus posiciones
    matches = []
    for keyword in lower_sorted_keywords:
        for m in re.finditer(re.escape(keyword), text_lower):
            matches.append((m.start(), m.end(), text_content[m.start():m.end()]))

    # Ordenar las coincidencias por su posición inicial y eliminar superposiciones para evitar formato doble
    # Esto es una simplificación; un manejo robusto de superposiciones es más complejo.
    # Para este caso de uso, donde las frases clave son distintas, debería funcionar bien.
    matches.sort()

    # Lista para almacenar las coincidencias finales sin superposiciones problemáticas
    final_matches = []
    if matches:
        final_matches.append(matches[0])
        for i in range(1, len(matches)):
            # Si la coincidencia actual se superpone con la última añadida, ignorarla o resolver conflicto
            # En este caso, si la nueva coincidencia empieza antes de que termine la anterior, la ignoramos
            # para dar preferencia a la más larga/primera encontrada.
            if matches[i][0] >= final_matches[-1][1]: # Si no hay superposición, o es adyacente
                final_matches.append(matches[i])

    last_end = 0
    for start, end, matched_text in final_matches:
        # Add text before the matched keyword
        if start > last_end:
            p.add_run(text_content[last_end:start])

        # Add the bold keyword
        bold_run = p.add_run(matched_text)
        bold_run.bold = True
        last_end = end

    # Add any remaining text after the last bold keyword
    if last_end < len(text_content):
        p.add_run(text_content[last_end:])

    # If no matches were found, add the entire text without bold
    if not p.runs and text_content: # Asegurarse de que si no hay runs pero hay texto, se añada
        p.add_run(text_content)


# --- Helper Functions for Image Handling (sin cambios) ---
def add_image_with_fixed_dimensions(doc, img_url, add_page_break_before=True,
                                    target_width_inches=TARGET_IMAGE_WIDTH_INCHES,
                                    target_height_inches=TARGET_IMAGE_HEIGHT_INCHES):
    """
    Adds an image to the Word document, forcing it to specified target dimensions.
    This might stretch/compress the image or upscale it if it's smaller.
    `add_page_break_before`: If True, a page break is added before the image.
    """
    try:
        img_data = requests.get(img_url).content
        image = Image.open(io.BytesIO(img_data)).convert("RGB")

        new_width_px = int(target_width_inches * DPI)
        new_height_px = int(target_height_inches * DPI)

        image = image.resize((new_width_px, new_height_px), Image.Resampling.LANCZOS)

        img_stream = io.BytesIO()
        image.save(img_stream, format="PNG")
        img_stream.seek(0)

        if add_page_break_before:
            doc.add_page_break()

        doc.add_picture(img_stream, width=Inches(target_width_inches))

    except Exception as e:
        doc.add_paragraph(f"[Error al insertar imagen con dimensiones fijas: {img_url}. Error: {e}]")


def add_image_inline(doc, img_url, max_width_inches=INLINE_IMAGE_MAX_WIDTH_INCHES):
    """
    Adds an image to the Word document inline with text, with an optional maximum width.
    The image will maintain its aspect ratio and will only scale down if larger than max_width.
    """
    try:
        img_data = requests.get(img_url).content
        image = Image.open(io.BytesIO(img_data)).convert("RGB")
        original_width_px, original_height_px = image.size

        max_width_px = int(max_width_inches * DPI)

        if original_width_px > max_width_px:
            scale_factor = max_width_px / original_width_px
            new_width_px = max_width_px
            new_height_px = int(original_height_px * scale_factor)
            image = image.resize((new_width_px, new_height_px), Image.Resampling.LANCZOS)

        img_stream = io.BytesIO()
        image.save(img_stream, format="PNG")
        img_stream.seek(0)

        doc.add_picture(img_stream, width=Inches(image.width / DPI))

    except Exception as e:
        doc.add_paragraph(f"[Error al insertar imagen inline: {img_url}. Error: {e}]")


def is_placeholder_image(img_url, min_dim_px=MIN_IMAGE_DIMENSION_PX):
    """
    Checks if an image URL points to a likely placeholder based on its dimensions.
    Returns True if it's likely a placeholder, False otherwise.
    """
    try:
        img_data = requests.get(img_url, stream=True, timeout=5).content
        image = Image.open(io.BytesIO(img_data))
        width, height = image.size
        if width < min_dim_px or height < min_dim_px:
            return True
    except Exception as e:
        pass
    return False


# --- Main Logic (ligeras modificaciones en add_paragraph_with_bold_keywords llamadas) ---
def html_to_docx(html_path, output_path):
    """
    Converts an HTML file to a .docx document, processing text and images.
    Ensures key phrases are followed by their associated images with fixed dimensions,
    skipping over small placeholder images.
    """
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    doc = Document()
#Hola que haceeeee
    sections = doc.sections
    for section in sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)

    processed_image_urls = set()
    general_img_insert_counter = 0

    all_elements = list(soup.body.descendants)
    i = 0

    while i < len(all_elements):
        element = all_elements[i]

        if getattr(element, 'name', None) == 'p' and element.get_text(strip=True):
            text = element.get_text().strip()

            if is_key_phrase(text):
                original_text = text
                text = clean_text_final(text)
                print(f"✅ LIMPIEZA:\n- Antes: '{original_text}' (len={len(original_text)})\n- Después: '{text}' (len={len(text)})")

                # USAR add_paragraph_with_bold_keywords para el texto del párrafo
                add_paragraph_with_bold_keywords(doc, text, WORDS_TO_BOLD)

                found_associated_image = False
                j = i + 1
                while j < len(all_elements):
                    next_element = all_elements[j]
                    if getattr(next_element, 'name', None) == 'img':
                        img_url = next_element.get('src')
                        if img_url and img_url.startswith("http"):
                            if img_url in processed_image_urls:
                                j += 1
                                continue

                            if is_placeholder_image(img_url):
                                j += 1
                                continue

                            print(f"🖼️ Insertando imagen asociada a frase clave: {img_url}")
                            add_image_with_fixed_dimensions(doc, img_url, add_page_break_before=True,
                                                            target_width_inches=TARGET_IMAGE_WIDTH_INCHES,
                                                            target_height_inches=TARGET_IMAGE_HEIGHT_INCHES)
                            processed_image_urls.add(img_url)
                            found_associated_image = True
                            i = j
                            break
                        else:
                            break
                    elif next_element.get_text(strip=True) or getattr(next_element, 'name', None) in ['p', 'ul', 'div', 'span']:
                        break
                    j += 1

                if not found_associated_image:
                    j_skip = i + 1
                    skip_count = 0
                    while j_skip < len(all_elements) and skip_count < 5:
                        next_element_to_check = all_elements[j_skip]
                        if getattr(next_element_to_check, 'name', None) == 'p' and not next_element_to_check.get_text(strip=True):
                            print(f"⏭️  Eliminando salto de línea vacío después de frase clave (índice {j_skip})")
                            i += 1
                            j_skip += 1
                            skip_count += 1
                            continue
                        break

            else: # Not a key phrase, just add the paragraph text
                # USAR add_paragraph_with_bold_keywords para el texto del párrafo
                add_paragraph_with_bold_keywords(doc, text, WORDS_TO_BOLD)

        elif getattr(element, 'name', None) == 'ul':
            for li in element.find_all('li'):
                text = li.get_text().strip()
                if is_key_phrase(text):
                    original_text = text
                    text = clean_text_final(text)

                # Para listas, creamos el párrafo de la lista y luego usamos add_paragraph_with_bold_keywords
                # para añadir el texto dentro de ese párrafo.
                p_list = doc.add_paragraph('• ', style='List Bullet')
                p = doc.add_paragraph(style='List Bullet')
                p.add_run('• ') # Agrega la viñeta

                # Lógica para aplicar negrita al texto del elemento de la lista
                # similar a add_paragraph_with_bold_keywords, pero sobre el 'p' existente
                current_pos_li = 0
                text_lower_li = text.lower()
                sorted_keywords_li = sorted(WORDS_TO_BOLD, key=len, reverse=True)
                lower_sorted_keywords_li = [k.lower() for k in sorted_keywords_li]

                matches_li = []
                for keyword_li in lower_sorted_keywords_li:
                    for m_li in re.finditer(re.escape(keyword_li), text_lower_li):
                        matches_li.append((m_li.start(), m_li.end(), text[m_li.start():m_li.end()]))

                matches_li.sort()
                final_matches_li = []
                if matches_li:
                    final_matches_li.append(matches_li[0])
                    for k in range(1, len(matches_li)):
                        if matches_li[k][0] >= final_matches_li[-1][1]:
                            final_matches_li.append(matches_li[k])

                last_end_li = 0
                for start_li, end_li, matched_text_li in final_matches_li:
                    if start_li > last_end_li:
                        p.add_run(text[last_end_li:start_li])

                    bold_run_li = p.add_run(matched_text_li)
                    bold_run_li.bold = True
                    last_end_li = end_li

                if last_end_li < len(text):
                    p.add_run(text[last_end_li:])

                if not p.runs and text: # Si no se añadió nada, asegúrate de que el texto esté
                    p.add_run(text)
                # Fin de la lógica de negrita para listas

        elif getattr(element, 'name', None) == 'img':
            img_url = element.get('src')
            if img_url and img_url.startswith("http") and img_url not in processed_image_urls:
                if is_placeholder_image(img_url):
                    i += 1
                    continue

                general_img_insert_counter += 1
                if general_img_insert_counter <= 2:
                    print(f"🖼️ Insertando imagen inline (general): {img_url} (Imagen #{general_img_insert_counter})")
                    add_image_inline(doc, img_url, max_width_inches=INLINE_IMAGE_MAX_WIDTH_INCHES)
                else:
                    print(f"🖼️ Insertando imagen a página completa (general): {img_url} (Imagen #{general_img_insert_counter})")
                    add_image_with_fixed_dimensions(doc, img_url, add_page_break_before=False,
                                                    target_width_inches=TARGET_IMAGE_WIDTH_INCHES,
                                                    target_height_inches=TARGET_IMAGE_HEIGHT_INCHES)
                processed_image_urls.add(img_url)

        i += 1

    for p_idx in range(len(doc.paragraphs) - 1, -1, -1):
        p = doc.paragraphs[p_idx]
        if p.text.strip() == "" and not any(run.text.strip() for run in p.runs) and not p.runs:
             # This block is difficult for direct removal. The final list comprehension is more reliable.
            pass

    doc.paragraphs[:] = [p for p in doc.paragraphs if p.text.strip() != "" or any(run.text.strip() for run in p.runs)]

    doc.save(output_path)
    print(f"✅ Documento generado: {output_path}")
    return output_path


# --- Execution Block ---
if __name__ == "__main__":
    output_document_paths = []
    try:
        from google.colab import files
        print("Esperando la subida de archivos HTML...")
        uploaded = files.upload()

        for name, content in uploaded.items():
            temp_html_path = os.path.join(OUTPUT_DIRECTORY, name)
            with open(temp_html_path, 'wb') as f:
                f.write(content)

            output_docx_path = os.path.join(OUTPUT_DIRECTORY, f"{os.path.splitext(name)[0]}.docx")
            generated_doc_path = html_to_docx(temp_html_path, output_docx_path)
            output_document_paths.append(generated_doc_path)

        print("\nDescargando documentos generados...")
        for doc_path in output_document_paths:
            files.download(doc_path)
        print("¡Proceso completado!")

    except ImportError:
        print("\nEl módulo 'google.colab.files' no está disponible. Este script está configurado para Google Colab.")
        print("Si deseas ejecutarlo localmente, modifica la sección de 'Execution Block' para leer archivos desde tu sistema de archivos.")
        # Example for local execution (uncomment and modify as needed):
        # print("Ejemplo de ejecución local (descomenta y adapta):")
        # local_html_file = "path/to/your/local/file.html"
        # local_output_docx = os.path.join(OUTPUT_DIRECTORY, "output_local.docx")
        # if os.path.exists(local_html_file):
        #    html_to_docx(local_html_file, local_output_docx)
        # else:
        #    print(f"Error: El archivo '{local_html_file}' no se encontró.")

Esperando la subida de archivos HTML...


Saving attach_419_15-07-2025_1445.pdf.doc to attach_419_15-07-2025_1445.pdf.doc
🖼️ Insertando imagen inline (general): http://diebold.godoworks.com/files/forms/images/419/419_diebold_nixdorf.jpg (Imagen #1)
🖼️ Insertando imagen inline (general): http://diebold.godoworks.com/files/forms/images/419/419_diebold_nixdorf_2.jpg (Imagen #2)
✅ LIMPIEZA:
- Antes: 'FOTO DE LA PARTE EXTERNA DEL ATM' (len=32)
- Después: 'FOTO DE LA PARTE EXTERNA DEL ATM' (len=32)
🖼️ Insertando imagen a página completa (general): http://diebold.godoworks.com/getImagews.php?id=Vmxkd1ExVXhaRWRoUkZwV1YwZG9VVlp0Y0c5T1JsSlpZMGMxVG1GNlZUSlZiRkpUVkdzeFZXRkVVbFZpYmtKWFdsWlZNVlpXVGxsalJUVlRVak5PTmxZd1VrTmpNVkp6VTJwV1QxZEZXbkZWYWtwT1QxRTlQUT09K1A=&cuenta=419 (Imagen #3)
🖼️ Insertando imagen a página completa (general): http://diebold.godoworks.com/getImagews.php?id=Vmxkd1ExVXhaRWRoUkZwV1YwZG9VVlp0Y0c5T1JsSlpZMGMxVG1GNlZUSlZiRkpUVkdzeFZXRkVVbFZpYmtKWFdsWlZNVlpXVGxsalJUVlhVbFZWZDFZd1VrTmpNVkp6VTJwV1QxZEZXbkZWYWtwT1QxRTlQUT09K1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

¡Proceso completado!
